In [1]:
import pandas as pd
import requests
import rasterio
import geopandas as gpd
import joblib
import math
import io
from shapely.geometry import Point

In [2]:
spatial_model = joblib.load("landslide_random_forest.pkl")

print("Model loaded successfully")

Model loaded successfully


In [26]:
dem_folder = r"C:\Users\workr\landslide Data\Slope data"

def get_elevation(latitude, longitude):

    lat_tile = math.floor(latitude)
    lon_tile = math.floor(longitude)

    lat_name = f"n{lat_tile:02d}" if lat_tile >= 0 else f"s{abs(lat_tile):02d}"
    lon_name = f"e{lon_tile:03d}" if lon_tile >= 0 else f"w{abs(lon_tile):03d}"

    filename = f"{lat_name}_{lon_name}_1arc_v3.tif"
    filepath = os.path.join(dem_folder, filename)

    if not os.path.exists(filepath):
        print("DEM tile not found:", filename)
        return None

    with rasterio.open(filepath) as src:

        value = list(src.sample([(longitude, latitude)]))[0][0]

        if src.nodata is not None and value == src.nodata:
            return None

        if value == -32767:
            return None

        return float(value)

In [31]:
import numpy as np

def get_slope(latitude, longitude):

    lat_tile = math.floor(latitude)
    lon_tile = math.floor(longitude)

    lat_name = f"n{lat_tile:02d}" if lat_tile >= 0 else f"s{abs(lat_tile):02d}"
    lon_name = f"e{lon_tile:03d}" if lon_tile >= 0 else f"w{abs(lon_tile):03d}"

    filename = f"{lat_name}_{lon_name}_1arc_v3.tif"
    filepath = os.path.join(dem_folder, filename)

    if not os.path.exists(filepath):
        print("DEM tile not found:", filename)
        return None

    with rasterio.open(filepath) as src:

        row, col = src.index(longitude, latitude)

        if row < 1 or row >= src.height - 1:
            return None

        if col < 1 or col >= src.width - 1:
            return None

        window = rasterio.windows.Window(
            col - 1,
            row - 1,
            3,
            3
        )

        elevation = src.read(1, window=window).astype(float)

        if src.nodata is not None:
            if (elevation == src.nodata).any():
                return None

        elevation[elevation == -32767] = np.nan

        if np.isnan(elevation).any():
            return None

        dx = 111320 * math.cos(math.radians(latitude)) * abs(src.res[0])
        dy = 111320 * abs(src.res[1])

        dz_dx = (
            elevation[1, 2] - elevation[1, 0]
        ) / (2 * dx)

        dz_dy = (
            elevation[2, 1] - elevation[0, 1]
        ) / (2 * dy)

        slope = math.degrees(
            math.atan(
                math.sqrt(
                    dz_dx**2 + dz_dy**2
                )
            )
        )

        return float(slope)

In [5]:
import glob
import os

rainfall_folder = r"C:\Users\workr\landslide Data\rainfall data"

rainfall_files = glob.glob(
    os.path.join(
        rainfall_folder,
        "IMERG-Final.CLIM.200006-202305.V07B*.tif"
    )
)

print(rainfall_files)

['C:\\Users\\workr\\landslide Data\\rainfall data\\IMERG-Final.CLIM.200006-202305.V07B.tif']


In [6]:
rainfall_file = rainfall_files[0]

def get_rainfall(latitude, longitude):

    with rasterio.open(rainfall_file) as src:

        value = list(
            src.sample([(longitude, latitude)])
        )[0][0]

        if src.nodata is not None and value == src.nodata:
            return None

    return float(value)

In [7]:
def get_land_cover(latitude, longitude):

    lat_start = math.floor(latitude / 3) * 3
    lon_start = math.floor(longitude / 3) * 3

    lat_name = f"N{lat_start:02d}" if lat_start >= 0 else f"S{abs(lat_start):02d}"
    lon_name = f"E{lon_start:03d}" if lon_start >= 0 else f"W{abs(lon_start):03d}"

    filename = f"ESA_WorldCover_10m_2021_v200_{lat_name}{lon_name}_Map.tif"

    url = (
        "https://esa-worldcover.s3.eu-central-1.amazonaws.com/"
        f"v200/2021/map/{filename}"
    )

    try:

        with rasterio.Env(AWS_NO_SIGN_REQUEST="YES"):

            with rasterio.open(url) as src:

                value = list(
                    src.sample([(longitude, latitude)])
                )[0][0]

        return int(value)

    except Exception as e:

        print("Land cover error:", e)

        return None

In [8]:
def get_clay(latitude, longitude):

    url = "https://maps.isric.org/mapserv?map=/map/clay.map"

    params = [
        ("SERVICE", "WCS"),
        ("VERSION", "2.0.1"),
        ("REQUEST", "GetCoverage"),
        ("COVERAGEID", "clay_0-5cm_mean"),
        ("FORMAT", "GEOTIFF_INT16"),
        ("SUBSETTINGCRS", "http://www.opengis.net/def/crs/EPSG/0/4326"),
        ("OUTPUTCRS", "http://www.opengis.net/def/crs/EPSG/0/4326"),
        ("SUBSET", f"X({longitude-0.002},{longitude+0.002})"),
        ("SUBSET", f"Y({latitude-0.002},{latitude+0.002})")
    ]

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("Clay error:", response.status_code)
        return None

    with rasterio.open(io.BytesIO(response.content)) as src:

        value = list(
            src.sample([(longitude, latitude)])
        )[0][0]

    return float(value) / 10

In [9]:
river_file = r"C:\Users\workr\landslide Data\river data\HydroRIVERS\HydroRIVERS_v10_as_shp\HydroRIVERS_v10_as.shp"

rivers = gpd.read_file(
    river_file,
    engine="pyogrio"
)

print("River segments:", len(rivers))

River segments: 1428959


In [22]:
def get_river_distance(latitude, longitude):

    zone = int((longitude + 180) / 6) + 1
    epsg = 32600 + zone

    min_lon = longitude - 2
    max_lon = longitude + 2
    min_lat = latitude - 2
    max_lat = latitude + 2

    nearby_rivers = rivers.cx[
        min_lon:max_lon,
        min_lat:max_lat
    ].copy()

    if len(nearby_rivers) == 0:
        return None

    point = gpd.GeoDataFrame(
        geometry=[Point(longitude, latitude)],
        crs="EPSG:4326"
    )

    nearby_rivers = nearby_rivers.to_crs(epsg)
    point = point.to_crs(epsg)

    distances = nearby_rivers.geometry.distance(
        point.geometry.iloc[0]
    )

    distance_m = distances.min()

    return distance_m / 1000

In [11]:
def predict_location(latitude, longitude):

    print("Getting environmental features...")

    elevation = get_elevation(latitude, longitude)
    slope = get_slope(latitude, longitude)
    rainfall = get_rainfall(latitude, longitude)
    land_cover = get_land_cover(latitude, longitude)
    clay = get_clay(latitude, longitude)
    river_distance = get_river_distance(latitude, longitude)

    features = pd.DataFrame([{
        "Elevation": elevation,
        "Slope": slope,
        "Annual_Rainfall": rainfall,
        "Land_Cover": land_cover,
        "Clay_Content": clay,
        "Distance_to_River_km": river_distance
    }])

    probability = spatial_model.predict_proba(features)[0][1] * 100

    print()
    print("Latitude:", latitude)
    print("Longitude:", longitude)
    print()
    print("Elevation:", round(elevation, 2), "m")
    print("Slope:", round(slope, 2), "degrees")
    print("Annual Rainfall:", round(rainfall, 2), "mm")
    print("Land Cover:", land_cover)
    print("Clay Content:", round(clay, 2), "%")
    print("Distance to River:", round(river_distance, 3), "km")
    print()
    print("Landslide Susceptibility:", round(probability, 2), "%")

    return probability

In [43]:
import os
import glob
import random
import math
import pandas as pd

dem_folder = r"C:\Users\workr\landslide Data\Slope data"

# Find all downloaded SRTM tiles
dem_files = glob.glob(
    os.path.join(dem_folder, "*_1arc_v3.tif")
)

print("SRTM tiles available:", len(dem_files))

# Get the geographic extent of each available tile
available_tiles = []

for filepath in dem_files:

    filename = os.path.basename(filepath)

    name = filename.replace("_1arc_v3.tif", "")

    lat_part = name[:3]
    lon_part = name[4:]

    if lat_part[0].lower() == "n":
        lat = int(lat_part[1:])
    else:
        lat = -int(lat_part[1:])

    if lon_part[0].lower() == "e":
        lon = int(lon_part[1:])
    else:
        lon = -int(lon_part[1:])

    available_tiles.append((lat, lon))

print("Available tile locations:", len(available_tiles))

SRTM tiles available: 94
Available tile locations: 94


In [44]:
import numpy as np

np.random.seed(42)

background_points = []

# Convert known landslides to array
positive_points = gsi[
    ["Latitude", "Longitude"]
].dropna().values


def distance_km(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = math.radians(lat1)
    lat2 = math.radians(lat2)

    dlat = lat2 - lat1
    dlon = math.radians(lon2 - lon1)

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    return 2 * R * math.asin(math.sqrt(a))


while len(background_points) < 50:

    # Select one of the actual available SRTM tiles
    tile_lat, tile_lon = random.choice(available_tiles)

    # Generate a point inside that 1° tile
    latitude = np.random.uniform(
        tile_lat + 0.05,
        tile_lat + 0.95
    )

    longitude = np.random.uniform(
        tile_lon + 0.05,
        tile_lon + 0.95
    )

    # Keep at least 10 km away from known landslides
    too_close = False

    for lat, lon in positive_points:

        if distance_km(
            latitude,
            longitude,
            lat,
            lon
        ) < 10:

            too_close = True
            break

    if too_close:
        continue

    # Make sure elevation exists
    elevation = get_elevation(
        latitude,
        longitude
    )

    if elevation is None:
        continue

    # Make sure land cover exists
    land_cover = get_land_cover(
        latitude,
        longitude
    )

    if land_cover is None:
        continue

    # Reject permanent water
    if land_cover == 80:
        continue

    background_points.append(
        (latitude, longitude)
    )


background_locations = pd.DataFrame(
    background_points,
    columns=["Latitude", "Longitude"]
)

background_locations["Actual_Landslide"] = 0

print("Background locations:", len(background_locations))

background_locations

Background locations: 50


,Latitude,Longitude,Actual_Landslide
0,27.387086,92.905643,0
1,8.190417,76.190395,0
2,26.591004,88.687265,0
3,12.600668,75.175544,0
4,11.583173,75.091805,0
5,12.596790,74.203472,0
6,24.108546,94.903997,0
7,18.919069,82.777558,0
8,14.159834,75.495659,0
9,8.282902,77.646270,0


In [45]:
background_results = []

for _, row in background_locations.iterrows():

    print("=" * 60)
    print(
        "Location:",
        round(row["Latitude"], 5),
        round(row["Longitude"], 5)
    )

    try:
        probability = predict_location(
            row["Latitude"],
            row["Longitude"]
        )

        background_results.append({
            "Latitude": row["Latitude"],
            "Longitude": row["Longitude"],
            "Actual_Landslide": 0,
            "Predicted_Susceptibility": float(probability)
        })

    except Exception as e:

        print("ERROR:", e)

        background_results.append({
            "Latitude": row["Latitude"],
            "Longitude": row["Longitude"],
            "Actual_Landslide": 0,
            "Predicted_Susceptibility": None
        })

background_results_df = pd.DataFrame(background_results)

background_results_df

Location: 27.38709 92.90564
Getting environmental features...

Latitude: 27.387086106962627
Longitude: 92.90564287576893

Elevation: 2107.0 m
Slope: 26.34 degrees
Annual Rainfall: 2375.16 mm
Land Cover: 10
Clay Content: 31.9 %
Distance to River: 1.202 km

Landslide Susceptibility: 36.5 %
Location: 8.19042 76.1904
Getting environmental features...

Latitude: 8.190416776398193
Longitude: 76.19039506830258

Elevation: 0.0 m
Slope: 0.0 degrees
Annual Rainfall: 1837.04 mm
Land Cover: 0
Clay Content: 0.0 %
Distance to River: 81.915 km

Landslide Susceptibility: 2.0 %
Location: 26.591 88.68727
Getting environmental features...

Latitude: 26.59100351056889
Longitude: 88.68726532001644

Elevation: 92.0 m
Slope: 1.04 degrees
Annual Rainfall: 2932.49 mm
Land Cover: 10
Clay Content: 23.0 %
Distance to River: 1.014 km

Landslide Susceptibility: 5.5 %
Location: 12.60067 75.17554
Getting environmental features...

Latitude: 12.600667605250141
Longitude: 75.17554447458683

Elevation: 96.0 m
Slope: 12.

,Latitude,Longitude,Actual_Landslide,Predicted_Susceptibility
0,27.387086,92.905643,0,36.5
1,8.190417,76.190395,0,2.0
2,26.591004,88.687265,0,5.5
3,12.600668,75.175544,0,24.5
4,11.583173,75.091805,0,2.5
5,12.596790,74.203472,0,2.5
6,24.108546,94.903997,0,5.5
7,18.919069,82.777558,0,0.5
8,14.159834,75.495659,0,64.5
9,8.282902,77.646270,0,0.0


In [46]:
background_results_df

,Latitude,Longitude,Actual_Landslide,Predicted_Susceptibility
0,27.387086,92.905643,0,36.5
1,8.190417,76.190395,0,2.0
2,26.591004,88.687265,0,5.5
3,12.600668,75.175544,0,24.5
4,11.583173,75.091805,0,2.5
5,12.596790,74.203472,0,2.5
6,24.108546,94.903997,0,5.5
7,18.919069,82.777558,0,0.5
8,14.159834,75.495659,0,64.5
9,8.282902,77.646270,0,0.0


In [47]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

# Combine positive and background results
positive_results = results_50_df[
    [
        "Latitude",
        "Longitude",
        "Actual_Landslide",
        "Predicted_Susceptibility"
    ]
]

negative_results = background_results_df[
    [
        "Latitude",
        "Longitude",
        "Actual_Landslide",
        "Predicted_Susceptibility"
    ]
]

validation_df = pd.concat(
    [positive_results, negative_results],
    ignore_index=True
)

# Remove failed predictions
valid = validation_df.dropna(
    subset=["Predicted_Susceptibility"]
).copy()

y_true = valid["Actual_Landslide"]

# 50% threshold
y_pred = (
    valid["Predicted_Susceptibility"] >= 50
).astype(int)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

roc_auc = roc_auc_score(
    y_true,
    valid["Predicted_Susceptibility"]
)

cm = confusion_matrix(y_true, y_pred)

print("===== EXTERNAL VALIDATION =====")
print()
print("Total locations:", len(validation_df))
print("Valid predictions:", len(valid))
print()
print("Accuracy :", round(accuracy * 100, 2), "%")
print("Precision:", round(precision * 100, 2), "%")
print("Recall   :", round(recall * 100, 2), "%")
print("F1 Score :", round(f1 * 100, 2), "%")
print("ROC-AUC  :", round(roc_auc, 4))

print()
print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print()
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

specificity = tn / (tn + fp)

print()
print(
    "Specificity:",
    round(specificity * 100, 2),
    "%"
)

===== EXTERNAL VALIDATION =====

Total locations: 100
Valid predictions: 97

Accuracy : 79.38 %
Precision: 81.4 %
Recall   : 74.47 %
F1 Score : 77.78 %
ROC-AUC  : 0.8898

Confusion Matrix:
[[42  8]
 [12 35]]

True Negatives : 42
False Positives: 8
False Negatives: 12
True Positives : 35

Specificity: 84.0 %


In [48]:
pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 470.4 kB/s eta 0:01:43
   ---------------------------------------- 0.5/48.9 MB 470.4 kB/s eta 0:01:43
   ---------------------------------------- 0.5/48.9 MB 470.4 kB/s eta 0:01:43
    --------------------------------------- 0.8/48.9 MB 479.0 kB/s eta 0:01:41
    --------------------------------------- 0.8/48.9 MB 479.0 kB/s eta 0:01:41
    --------------------------------------- 1.0/48.9 MB 497.2 kB/s eta 0:01:37
    --------------------------------------- 1.0/48.9 MB 497.2 kB/s eta 0:01:37
   - -------------------------------------- 1.3/48.9 MB 535.8 kB/s eta 0:01:29
   - -------------------------------------- 1.3/48.9 MB 535.8 kB/s eta 0:01:29
   - ----

In [49]:
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

preprocessor_xgb = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            ["Land_Cover"]
        ),
        (
            "num",
            "passthrough",
            [
                "Elevation",
                "Slope",
                "Annual_Rainfall",
                "Clay_Content",
                "Distance_to_River_km"
            ]
        )
    ]
)

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_xgb),
        (
            "classifier",
            XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                eval_metric="logloss",
                n_jobs=-1
            )
        )
    ]
)

In [51]:
import pandas as pd

data = pd.read_csv("landslide_hydrology_data.csv")

print("Data shape:", data.shape)

Data shape: (65712, 9)


In [52]:
data["Lat_Block"] = (data["Latitude"] // 1).astype(int)
data["Lon_Block"] = (data["Longitude"] // 1).astype(int)

data["Spatial_Block"] = (
    data["Lat_Block"].astype(str)
    + "_"
    + data["Lon_Block"].astype(str)
)

print("Spatial blocks:", data["Spatial_Block"].nunique())

Spatial blocks: 94


In [53]:
X = data[
    [
        "Elevation",
        "Slope",
        "Annual_Rainfall",
        "Land_Cover",
        "Clay_Content",
        "Distance_to_River_km"
    ]
]

y = data["Landslide"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (65712, 6)
y shape: (65712,)


In [54]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=data["Spatial_Block"]
    )
)

X_train_spatial = X.iloc[train_idx]
X_test_spatial = X.iloc[test_idx]

y_train_spatial = y.iloc[train_idx]
y_test_spatial = y.iloc[test_idx]

print("Training:", len(X_train_spatial))
print("Testing:", len(X_test_spatial))

Training: 50847
Testing: 14865


In [55]:
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

preprocessor_xgb = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            ["Land_Cover"]
        ),
        (
            "num",
            "passthrough",
            [
                "Elevation",
                "Slope",
                "Annual_Rainfall",
                "Clay_Content",
                "Distance_to_River_km"
            ]
        )
    ]
)

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_xgb),
        (
            "classifier",
            XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                eval_metric="logloss",
                n_jobs=-1
            )
        )
    ]
)

print("XGBoost model created")

XGBoost model created


In [56]:
xgb_model.fit(
    X_train_spatial,
    y_train_spatial
)

print("XGBoost training completed")

XGBoost training completed


In [57]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

xgb_pred = xgb_model.predict(
    X_test_spatial
)

xgb_probability = xgb_model.predict_proba(
    X_test_spatial
)[:, 1]

print("===== XGBOOST SPATIAL VALIDATION =====")
print()

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test_spatial,
            xgb_pred
        ) * 100,
        2
    ),
    "%"
)

print(
    "Precision:",
    round(
        precision_score(
            y_test_spatial,
            xgb_pred
        ) * 100,
        2
    ),
    "%"
)

print(
    "Recall:",
    round(
        recall_score(
            y_test_spatial,
            xgb_pred
        ) * 100,
        2
    ),
    "%"
)

print(
    "F1 Score:",
    round(
        f1_score(
            y_test_spatial,
            xgb_pred
        ) * 100,
        2
    ),
    "%"
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(
            y_test_spatial,
            xgb_probability
        ),
        4
    )
)

===== XGBOOST SPATIAL VALIDATION =====

Accuracy: 88.92 %
Precision: 89.87 %
Recall: 91.78 %
F1 Score: 90.82 %
ROC-AUC: 0.9449


In [58]:
import joblib

joblib.dump(
    xgb_model,
    "landslide_xgboost.pkl"
)

print("XGBoost model saved successfully")

XGBoost model saved successfully
